# Figure 5: PFT-level Boxplots of ET Components

Data source: 5-year averaged (2010-2014) CLM5 output from `G:/Hangkai/CTH_ET project/Final_data/`

In [ ]:
import netCDF4
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ============================================================
# Load 5-year averaged data from Final_data
# (averaged over 2010-2014, see NC_File_Averaging notebook)
# ============================================================
file_path = r'G:\Hangkai\CTH_ET project\Final_data'

scenario_files = {
    'CLM Default': 'CLM Default.nc',
    'GEDI Max':    'GEDI Max.nc',
    'GEDI Mean':   'GEDI Mean.nc',
    'GEDI Median': 'GEDI Median.nc',
}

# Scenario order for plotting (matches legend order)
scenarios = ['CLM Default', 'GEDI Max', 'GEDI Mean', 'GEDI Median']

et_variables = ['FCEV', 'FCTR', 'FGEV']

data = {}

for scenario in scenarios:
    data[scenario] = {}
    nc_file = os.path.join(file_path, scenario_files[scenario])
    with netCDF4.Dataset(nc_file, 'r') as ds:
        for var in et_variables:
            data[scenario][var] = ds.variables[var][:]  # shape: (12, 50430)

# Load PFT structure variables from any scenario (same across all)
with netCDF4.Dataset(os.path.join(file_path, 'CLM Default.nc'), 'r') as ds:
    pfts1d_itype_veg = np.array(ds.variables['pfts1d_itype_veg'][:])
    pfts1d_wtgcell   = np.array(ds.variables['pfts1d_wtgcell'][:])

# Handle potential time dimension in structural variables
if pfts1d_itype_veg.ndim > 1:
    pfts1d_itype_veg = pfts1d_itype_veg[0, :]
if pfts1d_wtgcell.ndim > 1:
    pfts1d_wtgcell = pfts1d_wtgcell[0, :]

# PFT names and IDs (tree PFTs 1-8)
pft_names = {
    "NET Temperate": 1,
    "NET Boreal": 2,
    "NDT Boreal": 3,
    "BET Tropical": 4,
    "BET Temperate": 5,
    "BDT Tropical": 6,
    "BDT Temperate": 7,
    "BDT Boreal": 8,
}

# Calculate annual sum for each scenario and PFT
annual_mean = {}
for scenario in scenarios:
    annual_mean[scenario] = {}
    for pft_name, pft_id in pft_names.items():
        pft_indexes = np.where(pfts1d_itype_veg == pft_id)[0]

        fcev = sum(data[scenario]['FCEV'][:, pft_indexes])  # sum across 12 months
        fgev = sum(data[scenario]['FGEV'][:, pft_indexes])
        fctr = sum(data[scenario]['FCTR'][:, pft_indexes])

        annual_mean[scenario][pft_name] = {
            'FCEV': fcev, 'FGEV': fgev, 'FCTR': fctr
        }

# Reformat data for boxplot
reformatted_data = []
for scenario in scenarios:
    for pft_name in pft_names:
        for variable in ['FCEV', 'FCTR', 'ET', 'FGEV']:
            if variable == 'ET':
                values = (annual_mean[scenario][pft_name]['FCEV'] +
                          annual_mean[scenario][pft_name]['FCTR'])
            else:
                values = annual_mean[scenario][pft_name][variable]
            for value in values:
                reformatted_data.append({
                    'Scenario': scenario,
                    'PFT': pft_name,
                    'Variable': variable,
                    'Value': float(value)
                })

reformatted_data = pd.DataFrame(reformatted_data)
print(f"Data loaded: {len(reformatted_data)} rows")
print(f"Scenarios: {scenarios}")

In [ ]:
# ============================================================
# Figure 5: PFT-level boxplots of ET components
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns

color_palette = sns.color_palette("pastel", n_colors=len(scenarios))
pft_order = list(pft_names.keys())

variables = ['FCEV', 'FGEV', 'FCTR', 'ET']
variable_labels = {'FCEV': 'Canopy EV', 'FGEV': 'Ground EV', 'FCTR': 'TR', 'ET': 'ET'}

fig, axs = plt.subplots(2, 2, figsize=(14, 10), dpi=300)
axs = axs.flatten()

for i, var in enumerate(variables):
    ax = axs[i]
    var_data = reformatted_data[reformatted_data['Variable'] == var]

    box_ax = sns.boxplot(x='PFT', y='Value', hue='Scenario', data=var_data,
                         palette=color_palette, ax=ax,
                         flierprops={"marker": "."}, fliersize=1)

    if i == 0:
        box_ax.legend(handles=box_ax.get_legend().legendHandles,
                      labels=['CLM Default', 'GEDI Max', 'GEDI Mean', 'GEDI Median'],
                      bbox_to_anchor=(0.00, 1.0), loc='upper left',
                      handlelength=1.5, handletextpad=0.5)
    else:
        box_ax.get_legend().remove()

    ax.set_xlabel("")
    ax.set_xticklabels(['NET Temp', 'NET Bor', 'NDT Bor', 'BET Trop', 'BET Temp',
                        'BDT Trop', 'BDT Temp', 'BDT Bor'], rotation=20, fontsize=12)

    for label in ax.get_yticklabels():
        label.set_fontsize(11)

    for box in box_ax.patches:
        box.set_edgecolor('gray')

axs[0].text(-0.15, 1.0, '(a)', transform=axs[0].transAxes, fontsize=16, va='top', ha='left')
axs[1].text(-0.15, 1.0, '(b)', transform=axs[1].transAxes, fontsize=16, va='top', ha='left')
axs[2].text(-0.15, 1.0, '(c)', transform=axs[2].transAxes, fontsize=16, va='top', ha='left')
axs[3].text(-0.15, 1.0, '(d)', transform=axs[3].transAxes, fontsize=16, va='top', ha='left')
axs[0].set_ylabel("Canopy Evaporation\n(W/m\u00b2)", fontsize=14)
axs[1].set_ylabel("Ground Evaporation\n(W/m\u00b2)", fontsize=14)
axs[2].set_ylabel("Transpiration\n(W/m\u00b2)", fontsize=14)
axs[3].set_ylabel("Evapotranspiration\n(W/m\u00b2)", fontsize=14)

plt.tight_layout()
plt.savefig('figures/Figure_5_PFT_Boxplots.png', dpi=300, bbox_inches='tight')
plt.savefig('figures/Figure_5_PFT_Boxplots.tiff', dpi=300, bbox_inches='tight')
plt.show()
print('Figure 5 saved.')